In [ ]:
# ============================================================
# NOTEBOOK 07B — SEQUENTIAL REPRESENTATION
# ALIGNED WITH 07A FOR FAIR MODEL COMPARISON
# ============================================================
#
# IMPORTANT:
# The sequential windows themselves are NOT modified.
# We first construct the valid 5x3 sequences exactly as before,
# then retain only sequence endpoints (UUID, timestamp) that are
# present in the final 07A tabular dataset.
#
# Thus:
# - sequence = [t-60, t-45, t-30, t-15, t]
# - features/step = temperature, relative_humidity, co2
# - target = temperature at t+15
# - no segment crossing
# - no split crossing
# - expected final counts = same as 07A
# ============================================================

from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path.cwd().resolve()
OUTPUTS = PROJECT_DIR / "outputs"

COMMON_FILE = OUTPUTS / "06_common_experiment" / "01_common_supervised_dataset.csv"
TABULAR_FILE = OUTPUTS / "07A_tabular" / "01_tabular_all_unscaled.csv"
OUT = OUTPUTS / "07B_sequential_aligned"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = ["temperature", "relative_humidity", "co2"]
TARGET = "target_temperature_t_plus_15"
STEP_MINUTES = 15
SEQUENCE_LENGTH = 5
HISTORY_MINUTES = 60

print("=" * 60)
print("NOTEBOOK 07B — SEQUENTIAL REPRESENTATION ALIGNED WITH 07A")
print("=" * 60)
print("Common input :", COMMON_FILE)
print("07A reference:", TABULAR_FILE)
print("Output       :", OUT)

assert COMMON_FILE.exists(), f"Missing: {COMMON_FILE}"
assert TABULAR_FILE.exists(), f"Missing: {TABULAR_FILE}"

# ------------------------------------------------------------
# 1. LOAD COMMON DATA
# ------------------------------------------------------------
df = pd.read_csv(COMMON_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["target_timestamp"] = pd.to_datetime(df["target_timestamp"], utc=True)

for c in FEATURES + [TARGET]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(["uuid", "segment_id", "timestamp"]).reset_index(drop=True)

# ------------------------------------------------------------
# 2. LOAD FINAL 07A SAMPLE KEYS
# ------------------------------------------------------------
tab = pd.read_csv(
    TABULAR_FILE,
    usecols=["uuid", "timestamp", "split", TARGET]
)
tab["timestamp"] = pd.to_datetime(tab["timestamp"], utc=True)

reference = tab[["uuid", "timestamp", "split", TARGET]].copy()
reference = reference.rename(columns={TARGET: "target_07A"})

assert reference.duplicated(["uuid", "timestamp"]).sum() == 0

print("\n07A reference counts:")
print(reference["split"].value_counts().reindex(
    ["train", "validation", "test"]
).to_string())

# ------------------------------------------------------------
# 3. BUILD VALID SEQUENCES EXACTLY AS BEFORE
# ------------------------------------------------------------
X_list, y_list, meta_rows = [], [], []

for (uuid, segment_id, split), g in df.groupby(
    ["uuid", "segment_id", "split"], sort=False
):
    g = g.sort_values("timestamp").reset_index(drop=True)

    values = g[FEATURES].to_numpy(dtype=np.float64)
    timestamps = g["timestamp"].tolist()
    targets = g[TARGET].to_numpy(dtype=np.float64)
    target_timestamps = g["target_timestamp"].tolist()

    for end_idx in range(SEQUENCE_LENGTH - 1, len(g)):
        start_idx = end_idx - SEQUENCE_LENGTH + 1
        ts_window = timestamps[start_idx:end_idx + 1]

        expected = [
            ts_window[0] + pd.Timedelta(minutes=STEP_MINUTES * k)
            for k in range(SEQUENCE_LENGTH)
        ]
        if ts_window != expected:
            continue

        if target_timestamps[end_idx] != (
            timestamps[end_idx] + pd.Timedelta(minutes=STEP_MINUTES)
        ):
            continue

        X_list.append(values[start_idx:end_idx + 1])
        y_list.append(targets[end_idx])
        meta_rows.append({
            "uuid": uuid,
            "segment_id": segment_id,
            "split": split,
            "sequence_start": timestamps[start_idx],
            "timestamp": timestamps[end_idx],
            "target_timestamp": target_timestamps[end_idx],
        })

X = np.stack(X_list).astype(np.float32)
y = np.asarray(y_list, dtype=np.float32)
meta = pd.DataFrame(meta_rows)

print("\nSequences before alignment:", len(meta))

# ------------------------------------------------------------
# 4. ALIGN ENDPOINTS WITH 07A
# ------------------------------------------------------------
# This filters complete sequences; it does NOT crop or alter them.

meta["_seq_index"] = np.arange(len(meta))

aligned = meta.merge(
    reference,
    on=["uuid", "timestamp", "split"],
    how="inner",
    validate="one_to_one"
).sort_values("_seq_index")

idx = aligned["_seq_index"].to_numpy(dtype=int)
X = X[idx]
y = y[idx]
meta = aligned.drop(columns=["_seq_index"]).reset_index(drop=True)

# Check that targets are exactly identical to 07A (within float precision)
if not np.allclose(
    y.astype(np.float64),
    meta["target_07A"].to_numpy(dtype=np.float64),
    rtol=0,
    atol=1e-5
):
    raise AssertionError("07A and 07B targets are not identical.")

meta = meta.drop(columns=["target_07A"])

# ------------------------------------------------------------
# 5. STRICT SEQUENCE INTEGRITY CHECKS
# ------------------------------------------------------------
duration = (
    pd.to_datetime(meta["timestamp"], utc=True)
    - pd.to_datetime(meta["sequence_start"], utc=True)
).dt.total_seconds() / 60

target_delta = (
    pd.to_datetime(meta["target_timestamp"], utc=True)
    - pd.to_datetime(meta["timestamp"], utc=True)
).dt.total_seconds() / 60

assert (duration == HISTORY_MINUTES).all()
assert (target_delta == STEP_MINUTES).all()
assert X.shape[1:] == (5, 3)
assert np.isfinite(X).all()
assert np.isfinite(y).all()

# ------------------------------------------------------------
# 6. CHECK EXACT SAME SAMPLE KEYS AS 07A
# ------------------------------------------------------------
seq_keys = set(zip(meta["uuid"], meta["timestamp"], meta["split"]))
tab_keys = set(zip(reference["uuid"], reference["timestamp"], reference["split"]))

assert seq_keys == tab_keys, (
    f"Alignment incomplete: sequential={len(seq_keys)}, "
    f"tabular={len(tab_keys)}"
)

# ------------------------------------------------------------
# 7. TRAIN-ONLY SCALING
# ------------------------------------------------------------
split_order = ["train", "validation", "test"]
indices = {
    s: np.flatnonzero(meta["split"].to_numpy() == s)
    for s in split_order
}

scaler = StandardScaler()
scaler.fit(
    X[indices["train"]].reshape(-1, len(FEATURES))
)

def scale_tensor(a):
    shape = a.shape
    return scaler.transform(
        a.reshape(-1, len(FEATURES))
    ).reshape(shape).astype(np.float32)

X_scaled = scale_tensor(X)

with (OUT / "sequential_standard_scaler.pkl").open("wb") as f:
    pickle.dump(scaler, f)

pd.DataFrame({
    "feature": FEATURES,
    "train_mean": scaler.mean_,
    "train_scale": scaler.scale_
}).to_csv(OUT / "07_scaler_train_statistics.csv", index=False)

# ------------------------------------------------------------
# 8. SAVE
# ------------------------------------------------------------
meta.to_csv(OUT / "01_all_sequence_metadata.csv", index=False)

audit = []
for s in split_order:
    ids = indices[s]
    part = meta.iloc[ids]

    np.save(OUT / f"X_{s}_unscaled.npy", X[ids])
    np.save(OUT / f"X_{s}_scaled.npy", X_scaled[ids])
    np.save(OUT / f"y_{s}.npy", y[ids])
    part.reset_index(drop=True).to_csv(
        OUT / f"metadata_{s}.csv", index=False
    )

    audit.append({
        "split": s,
        "sequences": len(ids),
        "devices": part["uuid"].nunique(),
        "sequence_start_min": part["sequence_start"].min(),
        "sequence_end_max": part["timestamp"].max(),
        "target_end_max": part["target_timestamp"].max()
    })

audit = pd.DataFrame(audit)
audit.to_csv(OUT / "02_sequential_split_audit.csv", index=False)

pd.DataFrame({
    "feature_index": range(len(FEATURES)),
    "feature": FEATURES
}).to_csv(OUT / "03_sequential_feature_manifest.csv", index=False)

protocol = {
    "representation": "sequential",
    "alignment_reference": "07A final tabular sample keys (UUID, timestamp, split)",
    "alignment_changes_sequence_content": False,
    "sequence_length": 5,
    "time_offsets_minutes": [-60, -45, -30, -15, 0],
    "history_minutes": 60,
    "features_per_step": FEATURES,
    "target": TARGET,
    "forecast_horizon_minutes": 15,
    "same_segment_required": True,
    "same_split_required": True,
    "scaler_fit": "training sequences only",
    "target_scaled": False,
    "fair_comparison": "same UUID/timestamp/target samples as 07A"
}
with (OUT / "04_sequential_protocol.json").open("w", encoding="utf-8") as f:
    json.dump(protocol, f, indent=2)

# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("FINAL ALIGNED COUNTS")
print("=" * 60)
print(audit.to_string(index=False))

print("\nTensor shapes:")
for s in split_order:
    ids = indices[s]
    print(f"{s:10s}: X={X[ids].shape}, y={y[ids].shape}")

print("\nVALIDATION")
print("✓ Sequences remain complete 5×3 tensors")
print("✓ Sequence content was NOT modified by alignment")
print("✓ History remains exactly 60 minutes")
print("✓ Target remains exactly t+15 minutes")
print("✓ No segment/split crossing")
print("✓ Same UUID/timestamp samples as 07A")
print("✓ Targets identical to 07A")
print("✓ Scaler re-fitted on aligned TRAIN only")

expected = {"train": 64706, "validation": 13825, "test": 13790}
actual = {s: len(indices[s]) for s in split_order}

print("\nExpected from current 07A:", expected)
print("Actual aligned 07B       :", actual)

assert actual == expected, (
    "Counts differ from the current 07A reference. "
    "Inspect the generated audit before training."
)

print("\n✓ 07B is exactly aligned with 07A.")
